# Recommendation System Evaluation

This notebook demonstrates comprehensive evaluation and visualization of Content-Based, Collaborative Filtering, and Hybrid recommendation models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_processing.data_loader import MovieLensDataLoader
from src.models.content_based import ContentBasedRecommender, ContentBasedConfig
from src.models.collaborative_filtering import CollaborativeFiltering
from src.models.hybrid import HybridRecommender
from src.evaluation.evaluator import RecommendationEvaluator
from src.evaluation.visualisation import RecommendationVisualizer
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

ModuleNotFoundError: No module named 'src.evaluation.visualization'

## 1. Load and Prepare Data

In [ ]:
loader = MovieLensDataLoader()
data_dict = loader.load_data()
await loader.letterboxd_data_async(max_concurrent_requests=500)

movies_df = pd.DataFrame(loader.movie_data)
ratings_df = data_dict['ratings']
genre_features = loader.preprocess_movies()
movies_df = pd.concat([movies_df, genre_features], axis=1)
movies_df = movies_df.dropna().reset_index(drop=True)

print(f"Movies: {len(movies_df)}")
print(f"Ratings: {len(ratings_df)}")
print(f"Users: {ratings_df['userId'].nunique()}")

## 2. Split Data (Temporal Split)

In [ ]:
ratings_df = ratings_df.sort_values('timestamp')
split_idx = int(len(ratings_df) * 0.8)

train_df = ratings_df.iloc[:split_idx].copy()
test_df = ratings_df.iloc[split_idx:].copy()

print(f"Train set: {len(train_df)} ratings")
print(f"Test set: {len(test_df)} ratings")
print(f"Train users: {train_df['userId'].nunique()}")
print(f"Test users: {test_df['userId'].nunique()}")

## 3. Train Models

In [ ]:
cb_config = ContentBasedConfig(
    main_actor_weight=0.3,
    director_weight=0.3,
    cast_weight=0.3,
    keywords_weight=0.6,
    numerical_weight=0.1,
    similarity_threshold=0.15,
    top_k_default=20
)

cb_model = ContentBasedRecommender(config=cb_config)
cb_model.fit(movies_df=movies_df, ratings_df=train_df)
print("Content-Based model trained successfully!")

In [ ]:
cf_model = CollaborativeFiltering(k_components=50, random_state=42)
cf_model.fit(df_ratings=train_df)
print("Collaborative Filtering model trained successfully!")

In [ ]:
hybrid_model = HybridRecommender(
    cf_model=cf_model,
    cb_model=cb_model,
    alpha=0.5
)
hybrid_model.fit(movies_df=movies_df, ratings_df=train_df)
print("Hybrid model trained successfully!")

## 4. Evaluate Models

In [ ]:
models = {
    'Content-Based': cb_model,
    'Collaborative': cf_model,
    'Hybrid': hybrid_model
}

evaluator = RecommendationEvaluator(
    models=models,
    train_df=train_df,
    test_df=test_df,
    relevance_threshold=4.0,
    user_sample_size=None,
    random_state=42
)

results_df = evaluator.evaluate_all_models(
    k_values=[5, 10, 20],
    max_recommendations=20
)

print("Evaluation completed!")
print(f"Results shape: {results_df.shape}")

In [ ]:
results_df

## 5. Visualizations

In [ ]:
visualizer = RecommendationVisualizer(results_df)

### 5.1 Precision@K Trend

In [ ]:
fig = visualizer.plot_metric_trend('precision', figsize=(10, 6))
plt.show()

### 5.2 Recall@K Trend

In [ ]:
fig = visualizer.plot_metric_trend('recall', figsize=(10, 6))
plt.show()

### 5.3 NDCG@K Trend

In [ ]:
fig = visualizer.plot_metric_trend('ndcg', figsize=(10, 6))
plt.show()

### 5.4 Model Comparison at K=10

In [ ]:
fig = visualizer.plot_model_comparison(k=10, figsize=(12, 7))
plt.show()

### 5.5 Performance Heatmap at K=10

In [ ]:
fig = visualizer.plot_all_metrics_heatmap(k=10, figsize=(10, 8))
plt.show()

### 5.6 Coverage vs Novelty Trade-off

In [ ]:
fig = visualizer.plot_coverage_novelty_tradeoff(k=10, figsize=(10, 7))
plt.show()

### 5.7 Radar Chart Comparison

In [ ]:
fig = visualizer.plot_radar_chart(k=10, figsize=(10, 10))
plt.show()

## 6. Save All Plots

In [ ]:
visualizer.save_all_plots(output_dir="evaluation_plots")
print("All plots saved to evaluation_plots/")

## 7. Summary Statistics

In [ ]:
print("=" * 80)
print("EVALUATION SUMMARY (K=10)")
print("=" * 80)

k10_results = results_df[results_df['k'] == 10]

for model in k10_results['model'].unique():
    model_data = k10_results[k10_results['model'] == model].iloc[0]
    print(f"\n{model.upper()}:")
    print(f"  Precision@10: {model_data['precision']:.4f}")
    print(f"  Recall@10:    {model_data['recall']:.4f}")
    print(f"  NDCG@10:      {model_data['ndcg']:.4f}")
    print(f"  MAP@10:       {model_data['map']:.4f}")
    print(f"  MRR:          {model_data['mrr']:.4f}")
    print(f"  Novelty:      {model_data['novelty']:.4f}")
    print(f"  Coverage:     {model_data['coverage']:.4f}")

print("\n" + "=" * 80)